In [3]:
from urielplus.urielplus import URIELPlus

u_pre = URIELPlus()          
u_pre.integrate_bdproto()
u_pre.integrate_apics()
u_pre.integrate_saphon()
u_pre.integrate_ewave()
u_pre.integrate_grambank()

u_pre.set_aggregation('U')
u_pre.aggregate()

2026-05-11 14:01:42,057 - root - INFO - Importing BDPROTO from "bdproto_data.csv"....
2026-05-11 14:01:42,057 - root - INFO - Converting ISO 639-3 codes to Glottocodes....
2026-05-11 14:01:42,249 - root - INFO - Conversion to Glottocodes complete.
2026-05-11 14:02:16,873 - root - INFO - BDPROTO integration complete.
2026-05-11 14:02:16,877 - root - INFO - Importing APiCS from "apics_data.csv"....
2026-05-11 14:02:19,490 - root - INFO - APiCS integration complete.
2026-05-11 14:02:19,493 - root - INFO - Importing updated SAPHON from "saphon_data.csv"....
2026-05-11 14:02:20,605 - root - INFO - Updated SAPHON integration complete..
2026-05-11 14:02:20,606 - root - INFO - Importing eWAVE from "english_dialect_data.csv"....
2026-05-11 14:02:30,412 - root - INFO - eWAVE integration complete.
2026-05-11 14:02:30,413 - root - INFO - Importing Grambank from "grambank_data.csv"....
2026-05-11 14:03:06,677 - root - INFO - Grambank integration complete.
2026-05-11 14:03:06,679 - root - INFO - Cre

array([[[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       ...,

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [ 0.],
        [ 0.],
        [ 1.]],

       [[ 0.],
        [ 1.],
        [ 0.],
        ...,
        [ 0.],
        [ 0.],
        [ 0.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [ 0.],
        [ 0.],
        [ 0.]]])

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from pathlib import Path

from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform

from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import LabelEncoder

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

try:
    from fancyimpute import SoftImpute
    HAS_SOFTIMPUTE = True
except ImportError:
    HAS_SOFTIMPUTE = False

OUTDIR = Path("urielplus_analysis")
OUTDIR.mkdir(exist_ok=True)

sns.set_style("whitegrid")

# URIEL+ Typological Data Empirical Missingness Analysis 
Sections:
1. Load data, build DataFrame, export CSV
2. All factors that contribute to missingness
3. All factors that help infer a missing value
4. SoftImpute diagnosis — where it works and where it breaks

Dependencies:
urielplus, numpy, pandas, scipy, sklearn,
matplotlib, seaborn, missingno, fancyimpute

(1) Build dataframe as well as the csv file

In [4]:
X_raw = u_pre.data[1].squeeze()

X = X_raw.astype(np.float64).copy()
X[X == -1] = np.nan

df_typ = pd.DataFrame(
    X,
    index=u_pre.langs[1],
    columns=u_pre.feats[1]
)

print("DataFrame shape:", df_typ.shape)

csv_path = OUTDIR / "typological_data.csv"

df_typ.index.name = "language"
df_typ.to_csv(csv_path)

print(f"CSV saved to: {csv_path}")

DataFrame shape: (8171, 800)
CSV saved to: urielplus_analysis/typological_data.csv


(2) Exploratory: \
We find factors that might contribute to missingness, and according to how the URIEL+ is organized initially, we look at from 3 angles: 
1. which source database does the feature come from? 
2. does missingness differ between languages with different language properties, such as family, area, resourse level and database membership count 
3. 

(a) Source

In [11]:
from urielplus.urielplus import URIELPlus

def build_single_db(db_name):

    u = URIELPlus()

    if db_name == "BDProto":
        u.integrate_bdproto()

    elif db_name == "APiCS":
        u.integrate_apics()

    elif db_name == "Saphon":
        u.integrate_saphon()

    elif db_name == "eWAVE":
        u.integrate_ewave()

    elif db_name == "Grambank":
        u.integrate_grambank()

    else:
        raise ValueError(f"Unknown database: {db_name}")

    u.set_aggregation("U")
    u.aggregate()

    return u


u_bd  = build_single_db("BDProto")
u_ap  = build_single_db("APiCS")
u_sa  = build_single_db("Saphon")
u_ew  = build_single_db("eWAVE")
u_gb  = build_single_db("Grambank")

2026-05-11 15:00:21,400 - root - INFO - Importing BDPROTO from "bdproto_data.csv"....
2026-05-11 15:00:21,402 - root - INFO - Converting ISO 639-3 codes to Glottocodes....
2026-05-11 15:00:21,828 - root - INFO - Conversion to Glottocodes complete.
2026-05-11 15:00:54,486 - root - INFO - BDPROTO integration complete.
2026-05-11 15:00:54,487 - root - INFO - Creating union of data across sources....
2026-05-11 15:00:54,574 - root - INFO - Executing new BFS-based genetic imputation strategy...
2026-05-11 15:01:02,859 - root - INFO - Found 3003 root languages for genetic imputation.
2026-05-11 15:01:02,880 - root - INFO - Genetic imputation filled 37960 values.
2026-05-11 15:01:02,880 - root - INFO - Genetic imputation finished.
2026-05-11 15:01:02,900 - root - INFO - Aggregation complete for idx=1.
2026-05-11 15:01:03,456 - root - INFO - Importing APiCS from "apics_data.csv"....
2026-05-11 15:01:03,457 - root - INFO - Converting ISO 639-3 codes to Glottocodes....
2026-05-11 15:01:03,628 - 

In [ ]:
db_objects = {
    "BDProto":  u_bd,
    "APiCS":    u_ap,
    "Saphon":   u_sa,
    "eWAVE":    u_ew,
    "Grambank": u_gb
}

db_languages = {}
db_features  = {}

for db_name, u in db_objects.items():

    db_languages[db_name] = set(u.langs[1])

    db_features[db_name] = set(u.feats[1])

print("Languages per Database:")
for k, v in db_languages.items():
    print(f"{k:10s}: {len(v)}")

print("\nFeatures per Database:")
for k, v in db_features.items():
    print(f"{k:10s}: {len(v)}")

Languages per DB:
BDProto   : 7989
APiCS     : 7830
Saphon    : 7970
eWAVE     : 7868
Grambank  : 7951

Features per DB:
BDProto   : 289
APiCS     : 431
Saphon    : 289
eWAVE     : 529
Grambank  : 458


In [15]:
db_membership = pd.DataFrame(index=df_typ.index)

for db_name, langs in db_languages.items():

    db_membership[db_name] = [
        int(lang in langs)
        for lang in df_typ.index
    ]

db_membership["n_databases"] = (
    db_membership.sum(axis=1)
)

db_membership["miss_rate"] = (
    df_typ.isna().mean(axis=1)
)

db_membership.head()

,BDProto,APiCS,Saphon,eWAVE,Grambank,n_databases,miss_rate
language,,,,,,,
ghot1243,1,1,0,1,1,4,1.0
alum1246,1,1,0,1,1,4,1.0
arii1243,1,1,0,1,1,4,1.0
amal1242,1,1,0,1,1,4,1.0
arbe1236,1,1,0,1,1,4,1.0


In [16]:
corr_db_count = db_membership[
    ["n_databases", "miss_rate"]
].corr().iloc[0, 1]

print(
    "Correlation(n_databases, miss_rate):",
    round(corr_db_count, 4)
)

Correlation(n_databases, miss_rate): 0.2225


(b) feature: Typological Modality

In [13]:
def infer_modality(feature):

    if feature.startswith("S_"):
        return "syntax"

    elif feature.startswith("P_"):
        return "phonology"

    elif feature.startswith("M_"):
        return "morphology"

    elif feature.startswith("INV_"):
        return "inventory"

    else:
        return "other"


feat_meta = pd.DataFrame({
    "feature": df_typ.columns,
    "modality": [infer_modality(f) for f in df_typ.columns],
    "miss_rate": df_typ.isna().mean().values,
    "coverage": df_typ.notna().mean().values,
    "n_present": df_typ.notna().sum().values,
}).set_index("feature")

feat_meta.head()

,modality,miss_rate,coverage,n_present
feature,,,,
S_SVO,syntax,0.702484,0.297516,2431
S_SOV,syntax,0.702729,0.297271,2429
S_VSO,syntax,0.702729,0.297271,2429
S_VOS,syntax,0.702974,0.297026,2427
S_OVS,syntax,0.702607,0.297393,2430


In [14]:
modality_miss = (
    feat_meta.groupby("modality")["miss_rate"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={
        "mean": "avg_miss",
        "count": "n_features"
    })
    .sort_values("avg_miss", ascending=False)
)

modality_miss.round(3)

,avg_miss,std,min,max,n_features
modality,,,,,
syntax,0.910,0.116,0.549,0.993,474
phonology,0.867,0.048,0.821,0.992,30
morphology,0.855,0.125,0.697,0.993,133
inventory,0.750,0.045,0.731,0.991,163


(a) source

In [7]:
db_miss = (
    feat_meta.groupby("source_db")["miss_rate"]
    .agg(["mean", "min", "max", "count"])
    .rename(columns={
        "mean": "avg_miss",
        "count": "n_features"
    })
    .sort_values("avg_miss", ascending=False)
)

db_miss.round(3)

,avg_miss,min,max,n_features
source_db,,,,
Unknown,0.867,0.549,0.993,800


(b) language

In [8]:
lang_miss = df_typ.isna().mean(axis=1).rename("miss_rate")

lang_miss.describe()

count    8171.000000
mean        0.866553
std         0.167364
min         0.005000
25%         0.772500
50%         0.981250
75%         1.000000
max         1.000000
Name: miss_rate, dtype: float64

In [9]:
try:
    lang_info = u_pre.get_language_info()

    lang_meta = lang_info[
        ["family", "genus", "macroarea", "latitude", "longitude"]
    ].copy()

    lang_meta["miss_rate"] = lang_miss

except Exception:

    lang_meta = lang_miss.to_frame()

    lang_meta["family"] = "unknown"
    lang_meta["macroarea"] = "unknown"
    lang_meta["latitude"] = np.nan
    lang_meta["longitude"] = np.nan

lang_meta.head()

,miss_rate,family,macroarea,latitude,longitude
language,,,,,
ghot1243,1.0,unknown,unknown,NaN,NaN
alum1246,1.0,unknown,unknown,NaN,NaN
arii1243,1.0,unknown,unknown,NaN,NaN
amal1242,1.0,unknown,unknown,NaN,NaN
arbe1236,1.0,unknown,unknown,NaN,NaN


In [10]:
fam_miss = (
    lang_meta.groupby("family")["miss_rate"]
    .agg(["mean", "count"])
    .rename(columns={
        "mean": "avg_miss",
        "count": "n_langs"
    })
    .sort_values("avg_miss", ascending=False)
    .head(20)
)

fam_miss.round(3)

,avg_miss,n_langs
family,,
unknown,0.867,8171
